# Day 1 — 데이터 파이프라인

업비트(국내 암호화폐 거래소)에서 4종목의 가격 데이터를 수집하고,  
투자 분석에 쓰이는 지표들을 계산해서 차트로 시각화합니다.

---

## 암호화폐 기초 지식 (처음이라면 꼭 읽어두세요)

### 우리가 다룰 4종목
| 시세 코드 | 이름 | 특징 |
|-----------|------|------|
| KRW-BTC | 비트코인(Bitcoin) | 세계 최초의 암호화폐. 암호화폐 시장의 기준점 |
| KRW-ETH | 이더리움(Ethereum) | 스마트 계약 기능이 있는 2세대 코인 |
| KRW-SOL | 솔라나(Solana) | 거래 속도가 매우 빠른 코인 |
| KRW-XRP | 리플(XRP) | 은행 간 송금에 특화된 코인 |

> `KRW-BTC`에서 **KRW**는 한국 원화(₩)를 뜻해요.  
> 즉, "원화로 거래되는 비트코인"이라는 의미입니다.

### OHLCV 데이터란?
주식/코인 가격 데이터를 표현하는 표준 형식입니다.

| 컬럼 | 영어 | 뜻 |
|------|------|----|
| open | Open | 하루 중 **첫 번째** 거래 가격 (시가) |
| high | High | 하루 중 **가장 높았던** 가격 (고가) |
| low | Low | 하루 중 **가장 낮았던** 가격 (저가) |
| close | Close | 하루 중 **마지막** 거래 가격 (종가) |
| volume | Volume | 하루 동안 거래된 **총 수량** (거래량) |

→ 우리는 주로 **종가(close)** 를 기준으로 분석합니다.  
→ 종가가 그날의 "최종 결과"이기 때문입니다.

---
## 0. 라이브러리 불러오기

파이썬의 기본 기능만으로는 데이터 분석이 힘들기 때문에  
전문가들이 미리 만들어둔 도구(라이브러리)를 가져와서 씁니다.

| 라이브러리 | 비유 | 우리가 쓰는 이유 |
|-----------|------|------------------|
| pyupbit | 배달앱 | 업비트 서버에서 코인 가격을 대신 받아다 줌 |
| pandas | 엑셀 | 데이터를 표(DataFrame) 형태로 쉽게 다룰 수 있게 해줌 |
| numpy | 계산기 | 루트(√), 평균 등 수학 계산을 빠르게 해줌 |
| plotly | 파워포인트 차트 | 마우스로 조작 가능한 인터랙티브 차트를 만들어 줌 |
| time | 타이머 | 코드 실행을 잠깐 멈출 때 사용 (재시도 대기) |
| datetime | 달력 | 오늘 날짜를 가져올 때 사용 |

In [ ]:
import pyupbit                             # 업비트 코인 가격 데이터 수집
import pandas as pd                        # 데이터를 표(DataFrame)로 다루기
import numpy as np                         # 수학 계산 (루트, 표준편차 등)
import time                                # 재시도 사이에 잠깐 멈추기
import plotly.graph_objects as go          # 차트 요소(선, 마커 등) 하나씩 직접 만들기
from plotly.subplots import make_subplots  # 여러 차트를 한 화면에 배치하기
from datetime import date                  # 오늘 날짜 가져오기

print("라이브러리 불러오기 완료!")

---
## 1. 데이터 수집

### 업비트 API란?

API(Application Programming Interface)란 **두 프로그램이 서로 대화하는 방법**입니다.  
우리가 pyupbit를 통해 업비트 서버에 "BTC 가격 줘!"라고 요청하면,  
업비트 서버가 "여기!"하고 데이터를 돌려주는 방식입니다.

```
[내 컴퓨터] ──요청──▶ [업비트 서버] ──데이터──▶ [내 컴퓨터]
```

### 왜 재시도(retry) 로직이 필요할까?

인터넷 연결이 순간 끊기거나, 서버가 바쁠 때 요청이 실패할 수 있어요.  
→ 1번 실패했다고 포기하지 않고, **최대 3번까지 다시 시도**하도록 만듭니다.

### 왜 캐싱(cache) 로직이 필요할까?

같은 데이터를 여러 번 API로 받아오면 시간도 느리고, 서버에 부담도 줘요.  
→ **오늘 이미 받은 데이터는 저장해두고**, 다시 요청이 오면 저장된 걸 돌려줍니다.

In [ ]:
# ── 분석할 종목 리스트 ────────────────────────────────────────────
# 나중에 종목을 추가하거나 바꾸고 싶을 때 이 리스트만 수정하면 됩니다
TICKERS = ["KRW-BTC", "KRW-ETH", "KRW-SOL", "KRW-XRP"]

# ── 캐시 저장소 ─────────────────────────────────────────────────
# 한번 받아온 데이터를 임시로 저장해두는 딕셔너리
# 딕셔너리 = 단어장처럼 '키(key): 값(value)' 쌍으로 데이터를 저장하는 자료구조
# 구조 예시: { "KRW-BTC": {"date": "2026-05-15", "df": 데이터프레임} }
cache = {}

print(f"분석 종목: {TICKERS}")
print(f"캐시 저장소 준비: {cache}  ← 지금은 비어 있음")

In [ ]:
def call_upbit_api(ticker):
    """
    업비트 API를 딱 한 번 호출해서 OHLCV 데이터를 받아옵니다.
    이 함수는 재시도 없이 한 번만 시도합니다.
    재시도 로직은 fetch_with_retry() 함수에서 담당합니다.

    매개변수(Parameter):
        ticker : 종목 코드 (예: "KRW-BTC")

    반환값(Return):
        성공 시 → DataFrame (날짜별 시가/고가/저가/종가/거래량 표)
        실패 시 → None
    """
    # pyupbit.get_ohlcv() : 업비트에서 일봉 캔들 데이터를 가져오는 함수
    # count=200 : 최근 200일치를 요청
    # 왜 180이 아닌 200? → 이동평균 계산에 앞쪽 여유 데이터가 필요해서
    # 예) MA60(60일 평균)은 최소 60일치 데이터가 쌓여야 계산됨
    # 나중에 tail(180)으로 최근 180일만 잘라서 씀
    df = pyupbit.get_ohlcv(ticker, count=200)
    return df

In [ ]:
def fetch_with_retry(ticker, max_retries=3, delay=1.0):
    """
    API 호출에 실패해도 최대 max_retries번까지 다시 시도합니다.

    매개변수:
        ticker      : 종목 코드 (예: "KRW-BTC")
        max_retries : 최대 재시도 횟수 (기본값: 3번)
        delay       : 재시도 사이 대기 시간, 단위 초 (기본값: 1초)

    반환값:
        성공 시 → DataFrame
        3번 모두 실패 시 → None
    """
    # range(3) → 0, 1, 2 순서로 반복 → 총 3번 시도
    for attempt in range(max_retries):
        try:
            # call_upbit_api() 로 API 한 번 호출
            result = call_upbit_api(ticker)

            # 결과가 None이 아니면 (= 정상적으로 받아왔으면) 바로 반환
            if result is not None:
                return result

        except Exception as e:
            # Exception : 예상치 못한 오류가 생겼을 때 잡아서 처리하는 구문
            # e : 어떤 오류인지 내용이 담겨 있음
            print(f"  [{ticker}] 시도 {attempt + 1}/{max_retries} 실패: {e}")

        # 마지막 시도가 아니면 delay초 기다렸다가 재시도
        # attempt < max_retries - 1 → 마지막(2번째 인덱스)이 아닌 경우
        if attempt < max_retries - 1:
            print(f"  {delay}초 후 재시도...")
            time.sleep(delay)  # delay초(=1초) 동안 코드 실행 멈춤

    # 반복문이 끝날 때까지 return을 못 만났다 = 3번 모두 실패
    print(f"  [{ticker}] 3번 모두 실패. None 반환.")
    return None

In [ ]:
def get_data(ticker):
    """
    캐시를 먼저 확인하고, 없으면 API를 호출해서 데이터를 가져옵니다.
    오늘 이미 받은 데이터가 있으면 API를 다시 호출하지 않습니다.

    매개변수:
        ticker : 종목 코드 (예: "KRW-BTC")

    반환값:
        DataFrame 또는 None
    """
    # 오늘 날짜를 문자열로 가져오기 (예: '2026-05-15')
    # str() : 날짜 객체를 문자열로 변환해야 비교가 편해짐
    today = str(date.today())

    # ── 캐시 확인 ────────────────────────────────────────────────
    # 'ticker in cache' : 딕셔너리에 해당 키가 존재하는지 확인
    if ticker in cache:
        # 저장된 날짜가 오늘이면 → 캐시 데이터 그대로 반환
        if cache[ticker]["date"] == today:
            print(f"  [{ticker}] 캐시에서 불러옴 (API 호출 생략)")
            return cache[ticker]["df"]

    # ── API 호출 ────────────────────────────────────────────────
    # 캐시에 없거나 오래된 데이터면 새로 받아옴
    print(f"  [{ticker}] API 호출 중...")
    df = fetch_with_retry(ticker)

    # ── 캐시에 저장 ─────────────────────────────────────────────
    # 정상적으로 데이터를 받아왔을 때만 캐시에 저장
    if df is not None:
        cache[ticker] = {
            "date": today,  # 오늘 날짜 저장 (다음에 날짜 비교할 때 씀)
            "df":   df      # 받아온 데이터프레임 저장
        }
        print(f"  [{ticker}] 수집 완료 → 캐시에 저장됨")

    return df

In [ ]:
# ── 4종목 데이터 수집 ─────────────────────────────────────────────
# data : 종목별 데이터프레임을 담을 딕셔너리
# 예: data["KRW-BTC"] → 비트코인 180일치 OHLCV 표
data = {}

print("=== 데이터 수집 시작 ===")
for ticker in TICKERS:
    print(f"\n[{ticker}]")
    df = get_data(ticker)

    if df is not None:
        # tail(180) : 마지막 180개 행만 가져옴 = 최근 180일 데이터
        # .copy()   : 원본 데이터프레임을 건드리지 않도록 복사본 사용
        data[ticker] = df.tail(180).copy()
        print(f"  → {len(data[ticker])}일치 데이터 준비됨")
    else:
        print(f"  → 수집 실패 (분석에서 제외됨)")

print(f"\n=== 수집 완료: 총 {len(data)}종목 ===")

In [ ]:
# 비트코인 데이터 마지막 3행 미리보기
# .tail(3) : 마지막 3개 행만 보여줌
# index(날짜), open(시가), high(고가), low(저가), close(종가), volume(거래량) 컬럼 확인
print("비트코인(KRW-BTC) 데이터 미리보기 (최근 3일):")
data["KRW-BTC"].tail(3)

---
## 2. 기술 지표 계산

가격 데이터만 보면 추세를 파악하기 힘들어요.  
그래서 **지표**를 계산해서 숫자를 더 의미 있게 만듭니다.

---

### 📈 수익률 (Return)

가격이 얼마나 올랐는지 퍼센트(%)로 나타낸 값입니다.

```
수익률 = (현재 가격 - 과거 가격) / 과거 가격 × 100

예) 어제 100만 원 → 오늘 105만 원
    수익률 = (105 - 100) / 100 × 100 = +5%
```

---

### 📊 이동평균선 (Moving Average, MA)

**최근 N일간의 종가 평균값**을 연결한 선입니다.  
하루하루 가격의 출렁임을 부드럽게 만들어서 **전체적인 방향(추세)** 을 보여줍니다.

```
MA5  (5일 평균)  → 단기 추세 (약 1주일)
MA20 (20일 평균) → 중기 추세 (약 1달)
MA60 (60일 평균) → 장기 추세 (약 3달)
```

**읽는 법:**
- MA5가 MA60보다 **위**에 있으면 → 단기적으로 상승 중 (좋은 신호)
- MA5가 MA60보다 **아래**에 있으면 → 단기적으로 하락 중 (나쁜 신호)

---

### 📐 볼린저 밴드 (Bollinger Bands)

1980년대 존 볼린저가 만든 지표로, MA20을 중심으로 **위아래에 밴드**를 그립니다.  
밴드의 너비는 **변동성(얼마나 요동치는지)** 에 따라 달라집니다.

```
상단 밴드 = MA20 + (표준편차 × 2)
하단 밴드 = MA20 − (표준편차 × 2)
```

**표준편차**란? 값들이 평균에서 얼마나 흩어져 있는지를 나타내는 숫자입니다.  
- 가격이 매일 크게 오르내리면 → 표준편차 크다 → 밴드 넓어짐
- 가격이 거의 안 변하면 → 표준편차 작다 → 밴드 좁아짐

**읽는 법:**
- 가격이 **상단 밴드** 근처 → 너무 많이 올랐을 수 있다 (과매수)
- 가격이 **하단 밴드** 근처 → 너무 많이 내렸을 수 있다 (과매도)

---

### 🌪️ 연환산 변동성 (Annualized Volatility)

가격이 얼마나 요동치는지를 **1년 기준으로 환산한 값**입니다.  
높을수록 = 리스크가 크다 = 크게 오를 수도, 크게 떨어질 수도 있다.

```
연환산 변동성 = 일별 수익률의 표준편차 × √365 × 100
```

> √365를 곱하는 이유: 암호화폐는 주식과 달리 **365일 24시간** 거래됩니다.  
> (주식은 주말 제외 약 252일 → 주식 변동성 계산에는 √252 사용)

In [ ]:
def calc_returns(df):
    """
    일별 수익률을 계산해서 DataFrame에 새 컬럼으로 추가합니다.

    추가되는 컬럼:
        return_1d  : 어제 대비 오늘 수익률 (%)
        return_7d  : 7일 전 대비 오늘 수익률 (%)
        return_30d : 30일 전 대비 오늘 수익률 (%)
    """
    # pct_change(n) : n행 전 값 대비 현재 값의 변화율 계산
    # 예) 어제 종가 100, 오늘 종가 105 → pct_change(1) = 0.05 (5%)
    # * 100 : 0.05 같은 소수를 5.0% 같은 퍼센트로 변환
    df["return_1d"]  = df["close"].pct_change(1)  * 100
    df["return_7d"]  = df["close"].pct_change(7)  * 100
    df["return_30d"] = df["close"].pct_change(30) * 100

    # 처음 n행은 n일 전 데이터가 없으므로 NaN(빈값)으로 표시됨
    # NaN = Not a Number, 계산할 수 없는 빈값
    return df

In [ ]:
def calc_moving_averages(df):
    """
    이동평균선(MA)을 계산해서 DataFrame에 추가합니다.

    추가되는 컬럼:
        MA5  : 5일 이동평균 (단기)
        MA20 : 20일 이동평균 (중기)
        MA60 : 60일 이동평균 (장기)
    """
    # rolling(n) : 최근 n개 행을 묶어서 처리하는 '슬라이딩 윈도우'
    # 마치 창문을 하나씩 밀면서 창문 안의 평균을 구하는 방식
    # 예) rolling(5).mean() → 오늘 포함 최근 5일 종가의 평균
    # 데이터가 n개 쌓이기 전까지는 NaN 으로 표시됨
    df["MA5"]  = df["close"].rolling(5).mean()   # 1주일 평균
    df["MA20"] = df["close"].rolling(20).mean()  # 한 달 평균
    df["MA60"] = df["close"].rolling(60).mean()  # 세 달 평균

    return df

In [ ]:
def calc_bollinger_bands(df):
    """
    볼린저 밴드의 상단/하단 값을 계산해서 DataFrame에 추가합니다.
    MA20이 먼저 계산되어 있어야 합니다 (calc_moving_averages 다음에 호출).

    추가되는 컬럼:
        upper_band : 상단 밴드 = MA20 + 표준편차 × 2
        lower_band : 하단 밴드 = MA20 − 표준편차 × 2
    """
    # 20일 표준편차: 최근 20일 종가가 평균에서 얼마나 흩어져 있는지
    # rolling(20).std() : 최근 20일 값의 표준편차
    std_20 = df["close"].rolling(20).std()

    # 상단 밴드: 평균 + 표준편차 × 2
    # (통계적으로 95% 확률로 가격이 상단~하단 밴드 안에 있음)
    df["upper_band"] = df["MA20"] + (std_20 * 2)

    # 하단 밴드: 평균 - 표준편차 × 2
    df["lower_band"] = df["MA20"] - (std_20 * 2)

    return df

In [ ]:
def calc_volatility(df):
    """
    연환산 변동성(20일 기준)을 계산해서 DataFrame에 추가합니다.

    추가되는 컬럼:
        volatility_20d : 최근 20일 수익률 표준편차 × √252 (연환산, %)
    """
    # 1단계: 일별 수익률 (소수점 형태, 예: 5% → 0.05)
    daily_return = df["close"].pct_change(1)

    # 2단계: 최근 20일 수익률의 표준편차
    # 표준편차가 크다 = 가격이 매일 크게 출렁인다 = 변동성이 크다
    std_20d = daily_return.rolling(20).std()

    # 3단계: 일별 표준편차 → 연간 기준으로 환산
    # √252 = 스펙 명세 기준값 (연간 거래일 수)
    # * 100 : 소수 → 퍼센트로 변환
    df["volatility_20d"] = std_20d * np.sqrt(252) * 100

    return df

In [ ]:
def add_indicators(df):
    """
    수익률, 이동평균, 볼린저밴드, 변동성 지표를 모두 한 번에 추가합니다.
    위에서 만든 네 개의 함수를 순서대로 호출합니다.

    순서가 중요합니다:
        1. calc_returns()         - 수익률 (독립적)
        2. calc_moving_averages() - 이동평균 (MA20이 볼린저밴드에 필요)
        3. calc_bollinger_bands() - 볼린저밴드 (MA20 계산 후에 가능)
        4. calc_volatility()      - 변동성 (독립적)
    """
    # .copy() : 원본 데이터프레임을 변경하지 않도록 복사본에 작업
    df = df.copy()

    df = calc_returns(df)           # 수익률 컬럼 추가
    df = calc_moving_averages(df)   # MA5, MA20, MA60 컬럼 추가
    df = calc_bollinger_bands(df)   # upper_band, lower_band 컬럼 추가
    df = calc_volatility(df)        # volatility_20d 컬럼 추가

    return df

In [ ]:
# ── 전체 종목에 지표 추가 ─────────────────────────────────────────
# for 루프로 4종목 각각에 add_indicators() 함수 적용
print("=== 지표 계산 시작 ===")
for ticker in data:
    data[ticker] = add_indicators(data[ticker])
    print(f"  [{ticker}] 지표 추가 완료")

print("\n=== 지표 계산 완료 ===")

# 새로 추가된 컬럼만 뽑아서 확인
indicator_cols = ["close", "return_1d", "return_7d",
                  "MA5", "MA20", "MA60",
                  "upper_band", "lower_band", "volatility_20d"]
print("\n비트코인 지표 미리보기 (최근 3일):")
data["KRW-BTC"][indicator_cols].tail(3)

---
## 3. 현황 요약 출력

계산한 지표를 보기 좋은 표 형태로 출력합니다.

### 출력 목표
```
=== 암호화폐 현황 요약 (기준일: 2026-05-15) ===

종목          현재가           7일 수익률   30일 수익률  연환산 변동성
---------  --------------  ----------  ----------  -----------
KRW-BTC    89,250,000 원     +5.23%      -2.11%       72.3%
```

숫자를 읽기 좋게 만드는 **포맷팅 함수**를 먼저 만들고,  
그 다음 표 전체를 출력하는 함수를 만듭니다.

In [ ]:
def format_price(price):
    """
    가격 숫자를 '89,250,000 원' 형태의 문자열로 변환합니다.

    예) 89250000 → '89,250,000 원'
    """
    # f-string 포맷 설명:
    # {:,}  → 천 단위마다 쉼표 삽입 (89250000 → 89,250,000)
    # {:.0f} → 소수점 없이 정수로 표시
    # {:,.0f} → 두 가지 동시 적용
    return f"{price:,.0f} 원"

In [ ]:
def format_pct(value):
    """
    수익률 숫자를 '+5.23%' 형태의 문자열로 변환합니다.
    NaN(빈값)이면 'N/A'를 반환합니다.

    예) 5.234  → '+5.23%'
        -2.1   → '-2.10%'
        NaN    → 'N/A'
    """
    # pd.isna() : 값이 NaN(Not a Number, 빈값)인지 확인
    # 데이터가 부족하면 일부 수익률은 NaN이 됨 (7일치 없으면 7일 수익률 계산 불가)
    if pd.isna(value):
        return "N/A"

    # {:+.2f} : 양수면 + 기호 자동 추가, 소수점 2자리
    return f"{value:+.2f}%"

In [ ]:
def format_vol(value):
    """
    변동성 숫자를 '72.3%' 형태의 문자열로 변환합니다.
    NaN이면 'N/A'를 반환합니다.

    예) 72.345 → '72.3%'
        NaN    → 'N/A'
    """
    if pd.isna(value):
        return "N/A"
    # {:.1f} : 소수점 첫째 자리까지만 표시
    return f"{value:.1f}%"

In [ ]:
def print_summary(data):
    """
    전체 종목의 현황 요약 표를 출력합니다.
    format_price(), format_pct(), format_vol() 함수를 활용합니다.
    """
    today = str(date.today())
    print(f"=== 암호화폐 현황 요약 (기준일: {today}) ===")
    print()

    # 헤더 출력
    # f-string 정렬: '<숫자' 왼쪽 정렬 / '>숫자' 오른쪽 정렬
    print(f"{'종목':<12} {'현재가':>18} {'7일 수익률':>12} {'30일 수익률':>12} {'연환산 변동성':>13}")
    print("-" * 74)  # 구분선

    for ticker in data:
        # iloc[-1] : 마지막 행(= 가장 최근 날짜) 가져오기
        last = data[ticker].iloc[-1]

        # 각 값을 읽기 좋은 형태의 문자열로 변환
        price_str = format_price(last["close"])
        ret7_str  = format_pct(last["return_7d"])
        ret30_str = format_pct(last["return_30d"])
        vol_str   = format_vol(last["volatility_20d"])

        print(f"{ticker:<12} {price_str:>18} {ret7_str:>12} {ret30_str:>12} {vol_str:>13}")


# 함수 호출
print_summary(data)

---
## 4. 볼린저 밴드 차트 (2×2)

4종목을 한 화면에 2행 2열로 배치해서 한눈에 비교합니다.

### plotly 구조 이해하기

```
fig (전체 화면)
 ├── subplot (1,1)  ← KRW-BTC
 ├── subplot (1,2)  ← KRW-ETH
 ├── subplot (2,1)  ← KRW-SOL
 └── subplot (2,2)  ← KRW-XRP
```

각 subplot 안에 여러 **trace(선/도형)** 를 순서대로 쌓아서 넣습니다:
1. 볼린저 밴드 채우기 (반투명 파란색 영역) ← 가장 아래 레이어
2. 상단/하단 밴드 점선
3. 종가선
4. MA5 / MA20 / MA60 선 ← 가장 위 레이어

### 볼린저 밴드 채우기 원리

상단선과 하단선 사이를 색으로 채우려면,  
상단 → 끝 → 하단(역방향)으로 이어지는 **폐곡선**을 그려야 합니다.

```
날짜 순서:  1월1일 → 1월2일 → ... → 5월15일  (상단값)
                                  ↓
           5월15일 → ... → 1월2일 → 1월1일  (하단값, 역순)
                                  ↓
           다시 1월1일로 돌아오면 폐곡선 완성!
```

In [ ]:
def make_chart_title(ticker, df):
    """
    차트 제목 문자열을 만들어 반환합니다.
    7일 수익률이 NaN이면 'N/A'로 안전하게 대체합니다.

    반환 예시: 'KRW-BTC  |  89,250,000 원  (+5.23% 7d)'
    """
    last   = df.iloc[-1]          # 가장 최근 행
    price  = last["close"]
    ret_7d = last["return_7d"]

    # NaN 체크: 7일 수익률이 없는 경우 대비 (format_pct가 처리)
    ret_str = format_pct(ret_7d)

    return f"{ticker}  |  {price:,.0f} 원  ({ret_str} 7d)"

In [ ]:
def add_bollinger_fill(fig, df, row, col, show_legend):
    """
    볼린저 밴드 상단~하단 사이를 반투명 파란색으로 채웁니다.

    원리: 상단선 데이터를 앞으로, 하단선 데이터를 역순으로 이어붙여
          닫힌 도형(폐곡선)을 만들고 내부를 색으로 채움.

    매개변수:
        fig         : plotly Figure 객체
        df          : 해당 종목 데이터프레임
        row, col    : subplot 위치 (예: row=1, col=2)
        show_legend : 범례 표시 여부
    """
    # 날짜 리스트를 앞으로 + 뒤집어서 이어붙임 (폐곡선의 x좌표)
    # list(df.index)        → [1월1일, 1월2일, ..., 5월15일]
    # list(df.index[::-1])  → [5월15일, ..., 1월2일, 1월1일] (역순)
    x_fill = list(df.index) + list(df.index[::-1])

    # 상단 밴드값 + 하단 밴드값 역순 (폐곡선의 y좌표)
    y_fill = list(df["upper_band"]) + list(df["lower_band"][::-1])

    fig.add_trace(
        go.Scatter(
            x=x_fill,
            y=y_fill,
            fill="toself",                      # 폐곡선 내부를 색으로 채움
            fillcolor="rgba(0,100,255,0.1)",    # 반투명 파란색
                                                # rgba(빨강,초록,파랑,투명도)
                                                # 투명도: 0=완전투명, 1=불투명
            line=dict(color="rgba(0,0,0,0)"),   # 테두리선 완전 투명 (안 보이게)
            name="볼린저 밴드",
            showlegend=show_legend,
            legendgroup="band",                 # 같은 그룹끼리 범례에서 묶임
            hoverinfo="skip"                    # 마우스 올렸을 때 팝업 표시 안 함
        ),
        row=row, col=col
    )

In [ ]:
def add_band_lines(fig, df, row, col):
    """
    볼린저 밴드 상단선과 하단선을 점선으로 그립니다.
    범례는 fill 영역에서 대표로 표시하므로 여기선 showlegend=False.
    """
    # 상단 밴드선
    fig.add_trace(
        go.Scatter(
            x=df.index, y=df["upper_band"],
            line=dict(color="royalblue", width=0.8, dash="dot"),  # dash="dot" : 점선
            showlegend=False,
            name="상단 밴드"
        ),
        row=row, col=col
    )

    # 하단 밴드선
    fig.add_trace(
        go.Scatter(
            x=df.index, y=df["lower_band"],
            line=dict(color="royalblue", width=0.8, dash="dot"),
            showlegend=False,
            name="하단 밴드"
        ),
        row=row, col=col
    )

In [ ]:
def add_price_and_ma_lines(fig, df, row, col, show_legend):
    """
    종가선과 이동평균선(MA5, MA20, MA60)을 그립니다.

    색상 규칙:
        종가  → 검정 (가장 중요한 값이므로 눈에 띄게)
        MA5   → 주황 (단기, 가격에 가장 민감하게 반응)
        MA20  → 파랑 (중기)
        MA60  → 보라 (장기, 가장 느리게 반응)
    """
    # 종가선
    fig.add_trace(
        go.Scatter(
            x=df.index, y=df["close"],
            line=dict(color="black", width=1.5),
            name="종가",
            showlegend=show_legend,
            legendgroup="close"
        ),
        row=row, col=col
    )

    # 이동평균선 정보를 리스트로 묶어서 반복 처리 (중복 코드 줄이기)
    # [(컬럼명, 색상, 범례그룹), ...] 형태
    ma_settings = [
        ("MA5",  "orange", "ma5"),
        ("MA20", "blue",   "ma20"),
        ("MA60", "purple", "ma60"),
    ]

    # 튜플 언패킹: for col_name, color, legend_group in ma_settings
    # → 리스트의 각 항목 ("MA5", "orange", "ma5") 을 세 변수에 각각 할당
    for col_name, color, legend_group in ma_settings:
        fig.add_trace(
            go.Scatter(
                x=df.index, y=df[col_name],
                line=dict(color=color, width=1.2),
                name=col_name,
                showlegend=show_legend,
                legendgroup=legend_group
            ),
            row=row, col=col
        )

In [ ]:
def make_bollinger_chart(data):
    """
    4종목의 볼린저 밴드 차트를 2×2 형태로 만들어 반환합니다.
    위에서 만든 helper 함수들을 조합해서 사용합니다.

    반환값:
        plotly Figure 객체 → fig.show()로 화면에 출력 가능
    """
    # ── 각 차트 제목 만들기 ──────────────────────────────────────
    # 리스트 컴프리헨션: 한 줄로 리스트를 만드는 파이썬 문법
    # [make_chart_title(t, data[t]) for t in data]
    # = for t in data: titles.append(make_chart_title(t, data[t])) 와 동일
    titles = [make_chart_title(t, data[t]) for t in data]

    # ── 2×2 subplot 틀 만들기 ────────────────────────────────────
    fig = make_subplots(
        rows=2, cols=2,           # 2행 2열 = 4칸
        subplot_titles=titles,    # 각 칸 제목
        vertical_spacing=0.12,    # 위아래 칸 사이 여백 비율 (0~1)
        horizontal_spacing=0.08   # 좌우 칸 사이 여백 비율
    )

    # ── 각 종목을 해당 칸에 배치 ─────────────────────────────────
    # enumerate : 인덱스(i)와 값(ticker)을 동시에 가져옴
    # i=0 → KRW-BTC, i=1 → KRW-ETH, ...
    positions = [(1,1), (1,2), (2,1), (2,2)]  # (행, 열) 위치

    for i, ticker in enumerate(data):
        df  = data[ticker]
        row = positions[i][0]   # 행 번호 (1 or 2)
        col = positions[i][1]   # 열 번호 (1 or 2)

        # 범례는 첫 번째 차트(i=0)에만 표시 (나머지는 중복이라 숨김)
        show_legend = (i == 0)

        # 레이어를 아래에서 위로 순서대로 쌓음
        add_bollinger_fill(fig, df, row, col, show_legend)   # 1. 채우기 (가장 아래)
        add_band_lines(fig, df, row, col)                    # 2. 밴드 점선
        add_price_and_ma_lines(fig, df, row, col, show_legend)  # 3. 종가 + MA (가장 위)

    # ── 전체 레이아웃 설정 ──────────────────────────────────────
    fig.update_layout(
        title="암호화폐 4종목 — 볼린저 밴드 차트 (최근 180일)",
        height=800,              # 전체 차트 높이 (픽셀)
        template="plotly_white"  # 흰 배경 테마
    )

    return fig

In [ ]:
# ── 차트 생성 & 출력 ──────────────────────────────────────────────
fig = make_bollinger_chart(data)
fig.show()  # 노트북 안에서 인터랙티브 차트 표시

# 사용 팁:
# - 마우스 드래그 → 특정 기간 확대
# - 더블클릭     → 원래 크기로 복원
# - 범례 클릭    → 해당 선 숨기기/보이기

---
## Day 1 완료 체크리스트

- [ ] 4종목 데이터 수집 완료 (180일치)
- [ ] 재시도(3회) + 캐싱 로직 동작 확인
- [ ] 지표 컬럼 추가 확인 (return_1d, return_7d, return_30d, MA5, MA20, MA60, upper_band, lower_band, volatility_20d)
- [ ] 현황 요약 표 정상 출력
- [ ] 볼린저 밴드 2×2 차트 정상 출력

---

## Day 2 예고

오늘 만든 `data` 딕셔너리와 함수들을 그대로 활용해서,  
**가상의 포트폴리오를 만들고** 90일간의 수익률을 추적합니다.